### WINDOWS FUNCTION

### Cumulative Sum

In [0]:
df.withColumn('totalsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

### RANK() and DENSE_RANK()

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()
     

### ROW_NUMBER()

In [0]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

### JOINS

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)
     


### INNER JOIN

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'inner').display()

### LEFT JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

### RIGHT JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

### ANTI JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

### WHEN - OTHERWISE

In [0]:
df.withColumn(
    "veg_exp_flag",
    when(
        (col("Veg_Flag") == "Veg") & (col("Item_MRP") < 100),
        "Veg_Inexpensive"
    ).when(
        (col("Veg_Flag") == "Veg") & (col("Item_MRP") >= 100),
        "Veg_Expensive"
    ).otherwise("Non-Veg")
).display()

In [0]:
df = df.withColumn("Veg_Flag", when(col("Item_Type") == "Meat", "Non-Veg").otherwise("Veg"))
df.display()

### PIVOT

In [0]:
df.groupBy("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

### COLLECT_LIST
PySpark aggregation function that collects values from multiple rows into an list.

In [0]:
df_book.groupBy("user").agg(collect_list("book")).display()

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

### GROUP_BY

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP"),avg("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type","Outlet_Size").sum("Item_MRP").display()

In [0]:
df.groupBy("Item_Type").avg("Item_MRP").display()

In [0]:
df.groupBy("Item_Type").sum("Item_MRP").display()

### ARRAY_CONTAINS

In [0]:
df_exp.withColumn("Type1_flag", array_contains("Outlet_Type","Type1")).display()

### EXPLODE
To get each value in new row rfom list

In [0]:
df_exp.withColumn("Outlet_Type", explode("Outlet_Type")).display()

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))

df_exp.display()
     


### SPLIT AND INDEXING
getItem() => We can allso use this to get index \
df.withColumn(
    "second_item",
    col("my_array_col").getItem(1)
).display()

In [0]:
df.withColumn("Outlet_Type", split(col("Outlet_Type"), " ")[1]).display()

In [0]:
df.withColumn("Outlet_Type", split(col("Outlet_Type"), " ")).display()


### HANDLING NULLS 
dropna("all") => drops complete row if all the values are null \
dropna("any") => drops all rows where we have null value available \
dropna(subset = ["Outlet_Size"]) => To drop rows by specifying particular column
fillna("NA") => This fills all column with NA where datatype is same as the value which we are using \
fillna(0,subset=["Item_Weight"]) => To fill nuLL value in particular column

In [0]:
df.fillna({
    "Item_Weight": 0,
    "Item_Visibility": 0,
    "Outlet_Size": "NA"
}).display()

In [0]:
df.fillna(0,subset=["Item_Weight"]).display()

In [0]:
df.fillna("NA",subset=["Outlet_Size"]).display()

In [0]:
df.fillna("NA").display()

In [0]:
df.dropna(subset = ["Outlet_Size"]).display()

In [0]:
df.dropna("any").display()

In [0]:
df.dropna("all").display()

### DATE_FORMAT \
to_date() → parsing (string → date) \
date_format() → formatting (date → string)

In [0]:
df = df.withColumn(
    "Week_Before",
    date_format(
        to_date(col("Week_Before"), "dd-MM-yyyy"),
        "dd-MM-yyyy"
    )
)
df.display()

### DATEDIFF

In [0]:
df = df.withColumn("Date_Diff" , date_diff(col("Week_After"), col("Week_Bef")))
df.display()

### DATE FUNCTIONS
CURRENT_DATE() \
DATE_ADD() \
DATE_SUB()

In [0]:
df = df.withColumn("Week_Bef", date_add(col("Current_Date"), -7))                         
df.display()

In [0]:
df = df.withColumn("Week_Before", date_sub(col("Current_Date"), 7))
df.display()

In [0]:
df = df.withColumn("Week_After", date_add(col("Current_Date"), 7))
df.display()

In [0]:
df = df.withColumn("Current_Date", current_date())
df.display()

### STRING Function
INITCAP() => Similar to excel Proper \
UPPER() \
LOWER()

In [0]:
df.select(initcap("Item_Type")).display()

### UNION AND UNION BYNAME

In [0]:

data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)


     


In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad','1',),
        ('sid','2',)]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)

df1.display()
     


In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

### DROP_DUPLICATES

In [0]:
df.distinct().display()

In [0]:
df.dropDuplicates(subset=["Item_Type"]).display()

In [0]:
df.dropDuplicates().display()

### Drop

In [0]:
df.drop("Item_Weight","Item_Fat_Content").display()

In [0]:
df.drop("Item_visibility").display()

### Limit
Works same as SQL

In [0]:
df.limit(10).display()

###Sort/OrderBy

In [0]:
df.sort(["Item_Weight", "Item_Visibility"], ascending=[False, True]).display()

In [0]:
df.sort(col("Item_Visibility").asc()).display()

### Type Casting

In [0]:
df = df.withColumn("Item_Weight", col("Item_Weight").cast(DoubleType()))

### withColumn/lit/regexp_replace
To modify or add new column based on calculation we can use this function &
lit function is used to store constant values in a column

In [0]:
df.withColumn("Item_Fat_Content", regexp_replace(col("Item_Fat_Content"), "Regular", "Reg")) \
    .withColumn("Item_Fat_Content", regexp_replace(col("Item_Fat_Content"), "Low Fat", "Lf")) \
        .display()

In [0]:
df.withColumn("Unit_Sold", round(col("Item_Outlet_Sales")/col("Item_MRP"),2)).display()

In [0]:
df = df.withColumn("flag",lit("new"))

### WithColumnRenamed
To change the column name to dataframe level

In [0]:
df.withColumnRenamed("Item_Weight","Item_Wt").display()

### FILTER/WHERE/ISNULL/ISIN

In [0]:
df.filter((col("Outlet_Size").isNull()) & (col("Outlet_Location_Type").isin(["Tier 1","Tier 2"])))\
  .display()

Multi Column Filter

In [0]:
df.filter((col("Item_Type")=="Soft Drinks") & (col("Item_Weight")<10)).display()

Single Column Filter

In [0]:
df.filter(col("Item_Fat_Content")=="Regular").display()

### ALIAS
It works same as sql AS

In [0]:
df.select(col("Item_Identifier").alias("Item_ID")).display()

### SELECT
It works same as sql

This used when we want to do something with columns like aggregation 

In [0]:
df_sel = df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

This used when just we want to select columns

In [0]:
df_sel = df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_strct_schema = StructType([
    StructField("Item_Identifier", StringType(), True),
    StructField("Item_Weight", DoubleType(), True),
    StructField("Item_Fat_Content", StringType(), True),
    StructField("Item_Visibility", DoubleType(), True),
    StructField("Item_Type", StringType(), True),
    StructField("Item_MRP", DoubleType(), True),
    StructField("Outlet_Identifier", StringType(), True),
    StructField("Outlet_Establishment_Year", IntegerType(), True),
    StructField("Outlet_Size", StringType(), True),
    StructField("Outlet_Location_Type", StringType(), True),
    StructField("Outlet_Type", StringType(), True),
    StructField("Item_Outlet_Sales", DoubleType(), True)
])

In [0]:
df = spark.read.format("csv") \
        .schema(my_strct_schema) \
        .option("header", "true") \
        .load("/Volumes/workspace/default/databricks_practise/BigMart Sales.csv")

df.display()

DDL Schema

In [0]:
df = spark.read.format("csv") \
        .schema(my_ddl_schema) \
        .option("header", "true") \
        .load("/Volumes/workspace/default/databricks_practise/BigMart Sales.csv")

In [0]:
my_ddl_schema = '''
                   Item_Identifier STRING,
                   Item_Weight STRING,
                   Item_Fat_Content STRING,
                   Item_Visibility DOUBLE,
                   Item_Type STRING,
                   Item_MRP DOUBLE,
                   Outlet_Identifier STRING,
                   Outlet_Establishment_Year Integer,
                   Outlet_Size STRING,
                   Outlet_Location_Type STRING,
                   Outlet_Type STRING,
                   Item_Outlet_Sales DOUBLE
                 '''


To change datatype of column we can use DDL and structType()
When to use which:
Quick notebook work	> DDL
Simple CSV schema	> DDL
Complex JSON	    > StructType
Nested arrays/structs	>  StructType
Production ETL	>  StructType
Dynamic schema generation	> StructType

printSchema() gives us datatype for the columns just like describe in Pandas

In [0]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



While reading json we don't have to specify inferschema() and header.
in json we don't have headers where as for json spark usually infers the schema automatically

In [0]:
df_json.display()

Reading json formatted file

In [0]:
df_json = spark.read.format("json") \
    .option("singleLine", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/databricks_practise/drivers.json")
df_json.show()


display() shows data in better tabluar format whereas show() shows somewhat messy

In [0]:
df.show() 

Checking all the available file in folder

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/databricks_practise"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/databricks_practise/BigMart Sales.csv,BigMart Sales.csv,869537,1778742004000
dbfs:/Volumes/workspace/default/databricks_practise/drivers.json,drivers.json,180812,1778792400000


Reading file from CSV

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/databricks_practise/BigMart Sales.csv")

display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window